In [ ]:
import json
import re
import urllib.parse
import requests
from bs4 import BeautifulSoup


class MovieInfoFetcher:

    def __init__(self):
        self.SESSION = requests.Session()
        self.HEADERS = {
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
        }

    def imdb_suggest(self, movie_name: str):
        q = movie_name.strip()
        if not q:
            return None, None

        path = f"{q[0].lower()}/{urllib.parse.quote(q)}.json"
        url = f"https://v2.sg.media-imdb.com/suggestion/{path}"
        r = self.SESSION.get(url, headers=self.HEADERS, timeout=15)

        if r.status_code != 200:
            return None, None

        data = r.json()

        # Pick first movie result (ID starts with 'tt')
        for item in data.get("d", []):
            id_ = item.get("id", "")
            if id_.startswith("tt"):
                title_txt = item.get("l") or item.get("q") or q
                return id_, title_txt
        return None, None

    def get_title_jsonld(self, tconst: str):
        url = f"https://www.imdb.com/title/{tconst}/"
        r = self.SESSION.get(url, headers=self.HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")

        # Find <script type="application/ld+json">
        ld = soup.find("script", type="application/ld+json")
        if not ld:
            return None

        try:
            jd = json.loads(ld.string)
            if isinstance(jd, list):
                for obj in jd:
                    if isinstance(obj, dict) and obj.get("@type") in ("Movie", "TVSeries", "CreativeWork"):
                        return obj
                return jd[0] if jd else None
            return jd
        except Exception:
            return None

    def extract_movie_details(self, jd):
        title = jd.get("name") or "N/A"

        # Year
        year = "N/A"
        date_pub = jd.get("datePublished")
        if date_pub and len(date_pub) >= 4:
            year = date_pub[:4]

        # Genres
        genres_raw = jd.get("genre")
        if isinstance(genres_raw, list):
            genres = ", ".join(genres_raw)
        elif isinstance(genres_raw, str):
            genres = genres_raw
        else:
            genres = "N/A"

        # Rating
        rating = "N/A"
        agg = jd.get("aggregateRating") or {}
        if isinstance(agg, dict) and agg.get("ratingValue"):
            rating = str(agg.get("ratingValue"))

        return {
            "title": title,
            "year": year,
            "rating": rating,
            "genres": genres
        }

    def trailer(self, movie_name: str):
        yq = urllib.parse.quote_plus(f"{movie_name} trailer")
        url = f"https://www.youtube.com/results?search_query={yq}"
        return url

    def fetch_movie(self, movie_name: str):
        tconst, display_title = self.imdb_suggest(movie_name)
        if not tconst:
            print("Movie not found on IMDb.")
            return

        jd = self.get_title_jsonld(tconst)
        if not jd:
            print(f"Could not parse details for {display_title} ({tconst}).")
            print(f"Title page: https://www.imdb.com/title/{tconst}/")
            return

        details = self.extract_movie_details(jd)

        # Print details
        print(f"\n🎞️ MOVIE DETAILS\n{'='*40}")
        print(f"Title: {details['title']}")
        print(f"Year: {details['year']}")
        print(f"IMDb Rating: {details['rating']}")
        print(f"Genre: {details['genres']}")
        print(f"IMDb Link: https://www.imdb.com/title/{tconst}/")
        print(f"\nTrailer: {self.trailer(movie_name)}")

if __name__ == "__main__":
    movie_name = input("Enter the movie name: ").strip()
    fetcher = MovieInfoFetcher()
    fetcher.fetch_movie(movie_name)

Enter the movie name: King
Could not parse details for Tulsa King (tt16358384).
Title page: https://www.imdb.com/title/tt16358384/


In [ ]:
#UPDATED - ISSUES WITH IMDB WEB SCRAPING.

import json
import urllib.parse
import requests
from bs4 import BeautifulSoup


class MovieInfoFetcher:
    def __init__(self):
        self.session = requests.Session()
        self.headers = {
            "User-Agent": "Mozilla/5.0"
        }

    def imdb_suggest(self, movie_name):
        q = movie_name.strip()

        if not q:
            return None

        path = f"{q[0].lower()}/{urllib.parse.quote(q)}.json"
        url = f"https://v2.sg.media-imdb.com/suggestion/{path}"

        try:
            response = self.session.get(url, headers=self.headers, timeout=15)
            response.raise_for_status()
            data = response.json()
        except:
            return None

        for item in data.get("d", []):
            movie_id = item.get("id", "")

            if movie_id.startswith("tt"):
                return item

        return None

    def get_title_jsonld(self, movie_id):
        url = f"https://www.imdb.com/title/{movie_id}/"

        try:
            response = self.session.get(url, headers=self.headers, timeout=20)
            response.raise_for_status()
        except:
            return None

        if "verify that you're not a robot" in response.text.lower():
            return None

        soup = BeautifulSoup(response.text, "html.parser")
        script = soup.find("script", type="application/ld+json")

        if not script:
            return None

        try:
            return json.loads(script.get_text(strip=True))
        except:
            return None

    def trailer(self, movie_name):
        query = urllib.parse.quote_plus(f"{movie_name} trailer")
        return f"https://www.youtube.com/results?search_query={query}"

    def fetch_movie(self, movie_name):
        movie = self.imdb_suggest(movie_name)

        if not movie:
            print("Movie not found on IMDb.")
            return

        movie_id = movie.get("id")
        title = movie.get("l", "N/A")
        year = movie.get("y", "N/A")

        data = self.get_title_jsonld(movie_id)

        rating = "N/A"
        genres = "N/A"

        if data:
            title = data.get("name", title)

            if data.get("datePublished"):
                year = data.get("datePublished")[:4]

            if isinstance(data.get("genre"), list):
                genres = ", ".join(data.get("genre"))
            elif data.get("genre"):
                genres = data.get("genre")

            rating_data = data.get("aggregateRating")
            if isinstance(rating_data, dict):
                rating = rating_data.get("ratingValue", "N/A")

        print("\nMOVIE DETAILS")
        print("=" * 40)
        print(f"Title: {title}")
        print(f"Year: {year}")
        print(f"IMDb Rating: {rating}")
        print(f"Genre: {genres}")
        print(f"IMDb Link: https://www.imdb.com/title/{movie_id}/")
        print(f"Trailer: {self.trailer(title)}")


if __name__ == "__main__":
    movie_name = input("Enter the movie name: ").strip()

    fetcher = MovieInfoFetcher()
    fetcher.fetch_movie(movie_name)